# SAM 3.1 Soccer GSR — Colab Runner (Phase 1)

Runs the tracking pipeline on a **Colab GPU** (a free T4 is enough for SAM 3, 848M params).

**Before running: Runtime → Change runtime type → GPU.**

Steps: verify GPU → clone repo → install deps → download gated `sam3.pt` → download one SoccerTrack v2 match → cut a short clip → run SAM 3 text-prompted **detect+track** → save `tracks.json` + an annotated `.mp4`.

Heavy logic lives in `src/` (per `CLAUDE.md §2`); this notebook just orchestrates and shows results.

> One-time in your browser: accept access at https://huggingface.co/facebook/sam3 so step 4 can download the weights.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clone (or pull) our repo, then cd into it
import os, getpass, subprocess

REPO_URL = "github.com/wheredawoodat949/AI-Hackathon.git"
REPO_DIR = "/content/AI-Hackathon"

if not os.path.isdir(REPO_DIR):
    # Private repo: paste a GitHub token (repo read scope). Leave blank if it's public.
    tok = getpass.getpass("GitHub token (blank if public): ").strip()
    url = f"https://{tok}@{REPO_URL}" if tok else f"https://{REPO_URL}"
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 3. Install deps. SAM 3 needs ultralytics >= 8.3.237.
!pip -q install -U "ultralytics>=8.3.237" gdown huggingface_hub python-dotenv pyyaml opencv-python-headless
# SAM 3 text prompts need the OpenAI CLIP package; safe to install up-front.
!pip -q install git+https://github.com/openai/CLIP.git

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
# 4. Download the GATED SAM 3 weights (sam3.pt) into the repo root.
#    Prereq (one-time, browser): accept access at https://huggingface.co/facebook/sam3
#    Then paste an HF token: https://huggingface.co/settings/tokens
import shutil, getpass, os
from huggingface_hub import login, hf_hub_download

login(getpass.getpass("HF token: ").strip())

# If this 404s, open the model page's "Files" tab and use the exact weight filename
# (e.g. a sam3.1 checkpoint) for `filename=` below.
src = hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt")
shutil.copy(src, "sam3.pt")
print("weights ->", os.path.abspath("sam3.pt"))

In [ ]:
# 5. Download ONE match from the SoccerTrack v2 Drive mirror (no HF gating).
#    Pulls gsr/bas/raw + the panoramic video for cfg.dev_match (117093). The 4K
#    video is large (a few GB) — that's expected; we cut a short clip next.
from src.config import load_config
from src.data.download import download_match

cfg = load_config()
root = download_match(cfg.dev_match, source="drive", include_videos=True)
print("data root:", root)

In [ ]:
# 6. Cut a short clip (cfg.clip_seconds). Downscale 4K->1920w for a fast first run;
#    drop the scale filter (or tile) later to recover small-player detail.
import glob, os, subprocess
from src.config import load_config

cfg = load_config()
cands = (glob.glob(str(cfg.data_root / "videos" / cfg.dev_match / "*.mp4"))
         or glob.glob(str(cfg.data_root / "videos" / f"{cfg.dev_match}*")))
assert cands, f"No video found under {cfg.data_root/'videos'} — check step 5."
src_vid = sorted(cands)[0]

os.makedirs(cfg.outputs, exist_ok=True)
clip = str(cfg.outputs / f"{cfg.dev_match}_clip.mp4")
subprocess.run(["ffmpeg", "-y", "-t", str(cfg.clip_seconds), "-i", src_vid,
                "-vf", "scale=1920:-2", "-an", clip], check=True)
print("source:", src_vid, "\nclip  :", clip)

In [ ]:
# 7. Run SAM 3 detect+track over the clip -> tracks.json + annotated mp4.
#    Requires sam.backend: local in config.yaml (weights run here on the GPU).
import json
from src.config import load_config
from src.utils.gpu import require_gpu, gpu_report
from src.model import get_backend
from src.tracking.annotate import annotate_video

print(gpu_report()); require_gpu()
cfg = load_config()
assert cfg.sam_backend == "local", "Set sam.backend: local in config.yaml for Colab."

backend = get_backend(cfg)
clip = str(cfg.outputs / f"{cfg.dev_match}_clip.mp4")
max_frames = cfg.clip_seconds * cfg.fps

frames = list(backend.track(clip, cfg.sam_prompts, max_frames=max_frames))
n_det = sum(len(f.detections) for f in frames)
print(f"frames={len(frames)}  detections={n_det}  "
      f"avg/frame={n_det/max(1,len(frames)):.1f}")

# structured output for Phase 2 (pitch/eval) + the frontend
tracks = [{"frame": f.frame_index,
           "detections": [{"id": d.instance_id, "label": d.label,
                           "score": round(d.score, 3),
                           "bbox": d.bbox_xyxy, "foot": d.foot_xy}
                          for d in f.detections]} for f in frames]
out_json = cfg.outputs / f"{cfg.dev_match}_tracks.json"
out_json.write_text(json.dumps(tracks))

out_mp4, n = annotate_video(clip, frames, cfg.outputs / f"{cfg.dev_match}_tracked.mp4")
print("tracks    ->", out_json)
print("annotated ->", out_mp4, f"({n} frames)")

In [ ]:
# 8. Show the annotated clip inline (re-encode to H.264 for the browser player).
import subprocess
from base64 import b64encode
from IPython.display import HTML
from src.config import load_config

cfg = load_config()
raw = str(cfg.outputs / f"{cfg.dev_match}_tracked.mp4")
h264 = str(cfg.outputs / f"{cfg.dev_match}_tracked_h264.mp4")
subprocess.run(["ffmpeg", "-y", "-i", raw, "-vcodec", "libx264",
                "-pix_fmt", "yuv420p", h264], check=True)

data = b64encode(open(h264, "rb").read()).decode()
HTML(f'<video width=960 controls src="data:video/mp4;base64,{data}"></video>')